# Preprocess Census Profile for MetroPopv1 Input Preparation

The Census profiles, 98-401-X20210, provide an output of all marginals publically provided by Statistics Canada at different levels of aggregation. This is a very large file comprising of roughly 2900 rows per geography, which generally span the whole country.

To reduce future run times, I have prepared this notebook to filter the full file into a reduced form that can be used for MetroPop input preparation and validation.

**Notes**

I am using the Aggregated Dissemination Area level of aggregation for the following reasons:
1. spans the full country unlike census tracts
2. similar in size to zone system
3. worried about suppression and rounding errors if I drop down to the dissemination area level.



### Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.options.display.max_columns = 60

### Inputs and data definitions

In [ ]:
working_dir = [set working directory]
fp = Path(working_dir) / r"98-401-X2021012_English_CSV_data.csv"

output_fp = Path(working_dir) / r"98-401-X2021012_English_CSV_data_trimmed_metropopv1.csv"

In [ ]:
# This marks the records that we'll keep in the Census population profile
# Note that I'm not keeping ID 1. This is because it's not broken down by gender
# Using column 8 for the total population instead.
id_ranges = [4, 5]                    # private dwellings
id_ranges.extend(range(8, 34))        # age breakdown
id_ranges.extend([39, 40])            # average and median age
id_ranges.extend(range(41, 50))       # dwelling type
id_ranges.extend(range(50, 58))       # household size
id_ranges.extend(range(89, 100))      # persons in private households
id_ranges.extend(range(100, 111))     # household type
id_ranges.extend([243])               # median total income of household
id_ranges.extend(range(260, 281))     # household total income
id_ranges.extend(range(2223, 2237))   # labour force status
id_ranges.extend(range(2246, 2259))   # NOCS classification
id_ranges.extend(range(2259, 2282))   # NAICS classification
id_ranges.extend(range(2593, 2598))   # PoW status
id_ranges.extend(range(2598, 2602))   # Commuting destination

In [ ]:
# Columns to drop
cols_to_drop = [
    'CENSUS_YEAR',           # always 2021
    'DGUID',                 # we're using ALT_GEO_CODE to store the ADA ID
    'GEO_LEVEL',             # always 'Aggregate dissemination area',
    'GEO_NAME',              # seems to be the same as ALT_GEO_CODE for this geometry type
                             # but ALT_GEO_CODE is an integer, so I prefer it.
    'CHARACTERISTIC_NOTE',   # description of field, refering to '98-401-X2021012_English_meta.txt' file
    # The following SYMBOL fields are usually blank
    'SYMBOL', 'SYMBOL.1', 'SYMBOL.2', 'SYMBOL.3', 'SYMBOL.4', 'SYMBOL.5',
    # We can't adjust rate columns to TAZs, only totals columns
    'C10_RATE_TOTAL', 'C11_RATE_MEN+', 'C12_RATE_WOMEN+'

]

# Notes: Keeping the following, even though there may not be a lot that we can do with these
# due to lack of other options. It's still good info when interpreting data.
#  TNSF, TNR_LF: which show the total non-response rate
#  DATA_QUALITY_FLAG': Data quality flag code, see metadata
# all count and rate columns: total, men+ and woment+

### Processing

In [ ]:
df = pd.read_csv(fp, encoding="ISO 8859-1")

In [ ]:
# Only keep columns of interest
df = df.drop(cols_to_drop, axis=1)

In [ ]:
# Only keep CHARACTERISTIC_IDs of interest
df = df.loc[df['CHARACTERISTIC_ID'].isin(id_ranges)]

In [ ]:
# Only keep ADAs in the model area
ada_fltr = ((df['ALT_GEO_CODE'] >= 35180000) & (df['ALT_GEO_CODE'] <= 35219999)) | (
            (df['ALT_GEO_CODE'] >= 35240000) & (df['ALT_GEO_CODE'] <= 35259999))
df = df.loc[ada_fltr]

In [ ]:
# That's it, export the file
df.to_csv(output_fp, encoding="ISO 8859-1")